In [7]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings("ignore")


df = pd.read_csv("FEATURES_PREPARED.csv")
df["Date"] = pd.to_datetime(df["Date"], utc=True)
df = df.sort_values("Date").reset_index(drop=True)

PM_FEATURES          = ["FED_DELTA", "GDP_DELTA", "UNEMPLOYMENT_DELTA", "INF_MONTHLY_DELTA"]
ANNOUNCEMENT_DUMMIES = ["Ann_CPI", "Ann_FOMC", "Ann_Employment", "Ann_GDP"]

ASSETS = {
    "BTC":       "BTC_chg",
    "Gold":      "Gold_chg",
    "Oil":       "Oil_chg",
    "SPY":       "SPY_chg",
    "QQQ":       "QQQ_chg",
    "SP500_fut": "SP500_fut_chg",
}
ASSETS = {k: v for k, v in ASSETS.items() if v in df.columns}
print(f"Assets available: {list(ASSETS.keys())}")

TC = {
    "SPY": 0.0002, "QQQ": 0.0002, "SP500_fut": 0.0001,
    "BTC": 0.0001, "Gold": 0.0001, "Oil": 0.0002,
}

HOLDING_PERIODS = {
    ("FED_DELTA",          "SPY"):       2,
    ("FED_DELTA",          "QQQ"):       2,
    ("FED_DELTA",          "SP500_fut"): 2,
    ("FED_DELTA",          "BTC"):       2,
    ("FED_DELTA",          "Gold"):      2,
    ("INF_MONTHLY_DELTA",  "Oil"):       1,
    ("UNEMPLOYMENT_DELTA", "SPY"):       1,
    ("UNEMPLOYMENT_DELTA", "QQQ"):       1,
    ("UNEMPLOYMENT_DELTA", "SP500_fut"): 1,
}
HOLDING_PERIODS = {k: v for k, v in HOLDING_PERIODS.items() if k[1] in ASSETS}

ANNUAL_HOURS   = 252 * 24
ROLLING_WINDOW = 20

print(f"Dataset: {df.shape[0]} rows, "
      f"{df['Date'].iloc[0].date()} → {df['Date'].iloc[-1].date()}")


print("\n" + "="*65)
print("  SHOCK CONSTRUCTION (with missingness correction)")
print("="*65)

for pm in PM_FEATURES:
    missing_col = f"{pm}_is_missing"

    pm_clean = df[pm].copy()
    if missing_col in df.columns:
        pm_clean[df[missing_col] == 1] = np.nan

    pm_clean_abs = pm_clean.abs()
    rolling_mean = pm_clean_abs.rolling(ROLLING_WINDOW, min_periods=5).mean().shift(1)
    rolling_std  = pm_clean_abs.rolling(ROLLING_WINDOW, min_periods=5).std().shift(1)
    threshold    = rolling_mean + 2 * rolling_std

    df[f"{pm}_shock"] = (df[pm].abs() > threshold.fillna(np.inf)).astype(int)
    df[f"{pm}_z"]     = (df[pm].abs() - rolling_mean) / (rolling_std + 1e-10)

    if missing_col in df.columns:
        df.loc[df[missing_col] == 1, f"{pm}_shock"] = 0

print("\nShock counts after missingness correction:")
for pm in PM_FEATURES:
    n = df[f"{pm}_shock"].sum()
    print(f"  {pm:30s}: {n} shocks ({100*n/len(df):.1f}%)")

ann_flag  = df[ANNOUNCEMENT_DUMMIES].any(axis=1)
any_shock = df[[f"{pm}_shock" for pm in PM_FEATURES]].any(axis=1)
overlap   = (ann_flag & any_shock).sum()
print(f"\n  Shock hours coinciding with official announcements: {overlap}")
print(f"  → {overlap}/{any_shock.sum()} = {100*overlap/max(any_shock.sum(),1):.1f}% of shocks")
print(f"  → Confirms most shocks are genuinely intraday, not announcement-driven")



# EVENT STUDY

print("\n\n" + "="*65)
print("  SECTION 3C — EVENT STUDY")
print("="*65)
print("""
For each scheduled macro announcement (CPI, FOMC, Employment, GDP):
  S_pre  = cumulative PM signal in PRE_HOURS before announcement
  R_post = cumulative asset return in POST_HOURS after announcement
  Regress R_post on S_pre (HC3 robust SEs).
  Significant β → PM was pricing in the announcement surprise beforehand.

Activity test: is PM more active in the PRE_HOURS before announcements
vs non-event hours? Ratio > 1 = PM participants are forward-looking.
""")

EVENT_PM_MAP = {
    "Ann_FOMC":       "FED_DELTA",
    "Ann_CPI":        "INF_MONTHLY_DELTA",
    "Ann_Employment": "UNEMPLOYMENT_DELTA",
    "Ann_GDP":        "GDP_DELTA",
}

PRE_HOURS  = 6
POST_HOURS = 3


print("Pre-announcement activity test (6 hours BEFORE announcement vs normal hours):")
print(f"  {'PM Signal':25s}  {'Event':20s}  {'n_events':>8}  {'Ratio':>8}  {'p-value':>10}")
print(f"  {'-'*78}")

activity_results = []
for ann_col, pm_col in EVENT_PM_MAP.items():
    ann_idx = df.index[df[ann_col] == 1].tolist()
    if len(ann_idx) < 3:
        continue

    pre_idx = set()
    for i in ann_idx:
        for j in range(max(0, i - PRE_HOURS), i):
            pre_idx.add(j)

    non_pre_idx = set(df.index) - pre_idx - set(ann_idx)

    event_vals  = df.loc[list(pre_idx),     pm_col].abs().dropna()
    normal_vals = df.loc[list(non_pre_idx), pm_col].abs().dropna()

    if len(event_vals) < 5 or len(normal_vals) < 5:
        continue

    ratio        = event_vals.mean() / (normal_vals.mean() + 1e-10)
    t_stat, pval = stats.ttest_ind(event_vals, normal_vals, equal_var=False)
    stars = "***" if pval<0.01 else "**" if pval<0.05 else "*" if pval<0.10 else ""
    print(f"  {pm_col:25s}  {ann_col:20s}  {len(ann_idx):8d}  "
          f"{ratio:8.2f}x  {pval:10.4f} {stars}")
    activity_results.append({
        "pm": pm_col, "event": ann_col, "n_events": len(ann_idx),
        "activity_ratio": round(ratio, 3), "pval_activity": round(pval, 4)
    })


# Predictive regression with HC3 robust SEs

print(f"\n\nPredictive regressions S_pre → R_post (HC3 robust standard errors):")
print(f"  {'Event':20s}  {'PM':25s}  {'Asset':10s}  {'n':>4}  {'β':>10}  {'p(HC3)':>10}")
print(f"  {'-'*88}")

event_study_results = []

for ann_col, pm_col in EVENT_PM_MAP.items():
    ann_hours = df.index[df[ann_col] == 1].tolist()
    if len(ann_hours) < 5:
        print(f"  {ann_col}: only {len(ann_hours)} events — skipping")
        continue

    for asset_name, asset_col in ASSETS.items():
        pre_signals, post_returns = [], []

        for idx in ann_hours:
            pre_window  = df.loc[max(0, idx - PRE_HOURS) : idx - 1, pm_col]
            s_pre       = pre_window.sum()

            post_end    = min(len(df) - 1, idx + POST_HOURS)
            post_window = df.loc[idx + 1 : post_end, asset_col]
            r_post      = post_window.sum()

            if pd.isna(s_pre) or pd.isna(r_post):
                continue
            if post_window.isna().all():
                continue

            pre_signals.append(s_pre)
            post_returns.append(r_post)

        if len(pre_signals) < 5:
            continue

        try:
            X     = sm.add_constant(pre_signals)
            model = sm.OLS(post_returns, X).fit(cov_type="HC3")  # HC3 robust SEs
            beta  = model.params[1]
            pval  = model.pvalues[1]
            n     = len(pre_signals)
            stars = ("***" if pval<0.01 else "**" if pval<0.05
                     else "*" if pval<0.10 else "~" if pval<0.15 else "")
            print(f"  {ann_col:20s}  {pm_col:25s}  {asset_name:10s}  "
                  f"{n:4d}  {beta:+10.4f}  {pval:10.4f} {stars}")
            event_study_results.append({
                "event": ann_col, "pm": pm_col, "asset": asset_name,
                "n_events": n, "beta": round(beta, 6), "pval_hc3": round(pval, 4)
            })
        except Exception:
            continue

if event_study_results:
    pd.DataFrame(event_study_results).to_csv("event_study_results.csv", index=False)
    print("\nEvent study results saved to: event_study_results.csv")


# SECTION 5 — BACKTEST & SHARPE RATIO

print("\n\n" + "="*65)
print("  SECTION 5 — BACKTEST & SHARPE RATIO")
print("="*65)
print("""
Strategy: trade on PM shock hours. Direction = sign(PM shock).
Hold for EXACTLY h hours (cumulative return over t+1 through t+h).
No overlapping trades: once a trade is open, skip new shocks until it closes.
Net Sharpe = gross Sharpe minus cost of round-trip transaction per trade.
""")


def compute_cumulative_return(df_in, asset_col, start_idx, h):
    """
    Sum of asset returns over rows start_idx+1 through start_idx+h (inclusive).
    Returns NaN if any hour in the window is missing (asset not trading).
    For 24/7 assets (BTC) this is almost never NaN.
    For SPY/QQQ it is NaN if the window crosses overnight/weekend gaps.
    """
    end_idx = min(start_idx + h, len(df_in) - 1)
    window  = df_in.loc[start_idx + 1 : end_idx, asset_col]
    if window.isna().any() or len(window) < h:
        return np.nan
    return window.sum()


def run_backtest(df_in, pm_col, asset_name, asset_col, h_hold=1, tc=0.0001):
    """
    Selective execution backtest with:
      - Cumulative h-hour holding period return (FIX 4)
      - No overlapping trades: skip shocks while a trade is open (FIX 5)
      - Direction = sign(PM shock) at decision time t
    """
    shock_rows  = df_in.index[df_in[f"{pm_col}_shock"] == 1].tolist()

    trade_returns = []
    trade_indices = []

    next_available = 0

    for idx in shock_rows:
        if idx < next_available:
            continue

        signal = np.sign(df_in.loc[idx, pm_col])
        if signal == 0:
            continue

        cum_ret = compute_cumulative_return(df_in, asset_col, idx, h_hold)
        if np.isnan(cum_ret):
            continue

        trade_returns.append(signal * cum_ret)
        trade_indices.append(idx)
        next_available = idx + h_hold

    if len(trade_returns) < 10:
        return None

    active   = np.array(trade_returns)
    n_trades = len(active)
    mean_ret = active.mean()
    std_ret  = active.std()

    gross_sr = np.sqrt(ANNUAL_HOURS) * mean_ret / (std_ret + 1e-10)
    net_sr   = np.sqrt(ANNUAL_HOURS) * (mean_ret - tc) / (std_ret + 1e-10)

    cum_curve = np.cumsum(active)
    peak      = np.maximum.accumulate(cum_curve)
    max_dd    = (cum_curve - peak).min()

    dir_acc = (active > 0).mean()
    n_pos   = int((active > 0).sum())
    binom_p = stats.binomtest(n_pos, n_trades, 0.5, alternative="greater").pvalue

    return {
        "pm": pm_col, "asset": asset_name, "h": h_hold,
        "n_trades": n_trades, "mean_ret": mean_ret,
        "cumulative_ret": active.sum(),
        "gross_sharpe": gross_sr, "net_sharpe": net_sr,
        "max_drawdown": max_dd,
        "dir_accuracy": dir_acc, "binom_p": binom_p,
        "_returns": active,
    }


print(f"  {'PM Signal':25s}  {'Asset':10s}  {'h':>2}  "
      f"{'Trades':>6}  {'Gross SR':>9}  {'Net SR':>8}  "
      f"{'MaxDD':>8}  {'DirAcc':>7}  {'p(dir)':>8}")
print(f"  {'-'*108}")

backtest_results = []
for (pm, asset_name), h in HOLDING_PERIODS.items():
    asset_col = ASSETS[asset_name]
    tc        = TC.get(asset_name, 0.0002)
    result    = run_backtest(df, pm, asset_name, asset_col, h_hold=h, tc=tc)
    if result is None:
        print(f"  {pm:25s}  {asset_name:10s}: too few trades — skipped")
        continue
    backtest_results.append(result)
    r     = result
    flag  = "✓" if r["net_sharpe"] > 0.5 else ("~" if r["net_sharpe"] > 0 else "✗")
    dstar = ("***" if r["binom_p"]<0.01 else "**" if r["binom_p"]<0.05
             else "*" if r["binom_p"]<0.10 else "")
    print(f"  {r['pm']:25s}  {r['asset']:10s}  {r['h']:2d}  "
          f"{r['n_trades']:6d}  "
          f"{r['gross_sharpe']:+9.3f}  {r['net_sharpe']:+8.3f} {flag}  "
          f"{r['max_drawdown']:+8.4f}  {r['dir_accuracy']:7.1%}  "
          f"{r['binom_p']:8.4f} {dstar}")


print("\nRandom signal benchmark (500 simulations — mean ± std):")
print(f"  {'PM → Asset':40s}  {'Mean Rand SR':>13}  {'Our Gross SR':>13}  {'Edge':>8}")
print(f"  {'-'*80}")

np.random.seed(42)
N_SIM = 500

for r in backtest_results:
    outcomes = r["_returns"]
    n        = len(outcomes)
    abs_rets = np.abs(outcomes)

    sim_srs = []
    for _ in range(N_SIM):
        rand_dir     = np.random.choice([-1, 1], size=n)
        rand_returns = rand_dir * abs_rets
        sim_sr       = np.sqrt(ANNUAL_HOURS) * rand_returns.mean() / (rand_returns.std() + 1e-10)
        sim_srs.append(sim_sr)

    mean_rand = np.mean(sim_srs)
    std_rand  = np.std(sim_srs)
    edge      = r["gross_sharpe"] - mean_rand
    label     = f"{r['pm']} → {r['asset']}"
    print(f"  {label:40s}  {mean_rand:+9.3f} ±{std_rand:.2f}  "
          f"{r['gross_sharpe']:+13.3f}  {edge:+8.3f}")

print("""
  Edge > 0: our strategy beats the average random signal
  Edge < 0: our strategy is worse than coin-flipping (signal in wrong direction)
  Note: random benchmark uses same trade magnitudes — only direction is randomised.
  This isolates whether our DIRECTION choice adds value.
""")


for r in backtest_results:
    r.pop("_returns", None)

if backtest_results:
    pd.DataFrame(backtest_results).to_csv("backtest_results.csv", index=False)
    print("Backtest results saved to: backtest_results.csv")


Assets available: ['BTC', 'Gold', 'Oil', 'SPY', 'QQQ', 'SP500_fut']
Dataset: 9673 rows, 2025-03-05 → 2026-04-12

  SHOCK CONSTRUCTION (with missingness correction)

Shock counts after missingness correction:
  FED_DELTA                     : 726 shocks (7.5%)
  GDP_DELTA                     : 712 shocks (7.4%)
  UNEMPLOYMENT_DELTA            : 538 shocks (5.6%)
  INF_MONTHLY_DELTA             : 609 shocks (6.3%)

  Shock hours coinciding with official announcements: 14
  → 14/2284 = 0.6% of shocks
  → Confirms most shocks are genuinely intraday, not announcement-driven


  SECTION 3C — EVENT STUDY

For each scheduled macro announcement (CPI, FOMC, Employment, GDP):
  S_pre  = cumulative PM signal in PRE_HOURS before announcement
  R_post = cumulative asset return in POST_HOURS after announcement
  Regress R_post on S_pre (HC3 robust SEs).
  Significant β → PM was pricing in the announcement surprise beforehand.

Activity test: is PM more active in the PRE_HOURS before announcements
vs 